# Lab 3.2: Content Moderation using Amazon Bedrock Data Automation

| Features | Amazon Bedrock <br/> Data Automation | Amazon Rekognition <br/> Content Moderation |
| --- | --- | --- |
| Modality | Image / Video | Image |
| Evaluated input | S3 bucket | Base64-encoded blob/S3 |
| Toxicity detection | [7 moderation categories](https://docs.aws.amazon.com/bedrock/latest/userguide/bda-ouput-image.html#content-moderation) | [3-level hierarchical categories](https://docs.aws.amazon.com/rekognition/latest/dg/moderation-api.html) |
| Toxicity dataset | Built-in | Built-in / Custom via adapters |
| Confidence score | Y | Y |
| API | [InvokeDataAutomationAsync](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomationAsync.html) <br/> [GetDataAutomationStatus](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomationAsync.html) | [DetectModerationLabels](https://docs.aws.amazon.com/rekognition/latest/APIReference/API_DetectModerationLabels.html) |
| Cost | Per image/video min | Per image |

## Pre-requisites

In [ ]:
import boto3
import json
import utils

In [ ]:
iam = boto3.client('iam')
sts = boto3.client('sts')
s3 = boto3.client('s3')
bda = boto3.client('bedrock-data-automation')
bda_runtime = boto3.client('bedrock-data-automation-runtime')
session = boto3.session.Session()

region = session.region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')

In [ ]:
from sagemaker import get_execution_role

policy_name = "CustomBDAPolicy"
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:CreateDataAutomationProject",
                "bedrock:GetDataAutomationProject",
                "bedrock:InvokeDataAutomationAsync"
            ],
            "Resource": [
                f"arn:aws:bedrock:{region}:{account_id}:data-automation-project/*"
            ]
        }
    ]
}

# Get current execution role
current_role = get_execution_role()
print(f"Current execution role: {current_role}")

role_name = current_role.split('/')[-1]
response = iam.put_role_policy(
    RoleName=role_name,
    PolicyName=policy_name,
    PolicyDocument=json.dumps(policy_document)
)
print(f"Successfully added inline policy {policy_name} to role {role_name}")

In [ ]:
bucket_name = account_id + "-" + region + "-" + "bda"

if region == "us-east-1":
    bucket_responese = s3.create_bucket(Bucket=bucket_name)
else:
    bucket_responese = s3.create_bucket(
        Bucket=bucket_name,
        CreateBucketConfiguration={'LocationConstraint': region}
        )
file_name = "man-smoking-cigarette.jpg"
file_location = f"images/{file_name}"
s3.upload_file(file_location, bucket_name, f"input/{file_name}")

## Create Bedrock Data Automation Project

In [ ]:
bda_project_arn = create_bda_project(
    project_name="marketing-project",
    project_description="Project for processing marketing images",
)
print(bda_project_arn)

## Run content moderation through Bedrock Data Automation

In [ ]:
input_s3_uri = f"s3://{bucket_name}/input/{file_name}" # File
output_s3_uri = f"s3://{bucket_name}/output" # Folder

params = {
    'inputConfiguration': {
        's3Uri': input_s3_uri
    },
    'outputConfiguration': {
        's3Uri': output_s3_uri
    },
    'dataAutomationConfiguration': {
        'dataAutomationArn': bda_project_arn,
        'stage': 'LIVE' #'DEVELOPMENT'|
    }#,
    #'dataAutomationProfileArn': f"arn:aws:bedrock:{region}:{account_id}:data-automation-profile/us.data-automation-v1"
}

response = bda_runtime.invoke_data_automation_async(**params)
invocation_arn = response['invocationArn']
print(response['invocationArn'])

In [ ]:
import time

while True:
    response = bda_runtime.get_data_automation_status(
        invocationArn=invocation_arn
    )
    status = response['status']
    if status not in ['Created', 'InProgress']:
        print(f" {status}")
        break 
    else:
        print(".", end='', flush=True)
        time.sleep(15)
print(json.dumps(response, indent=2))

In [ ]:
job_metadata_s3_uri = response['outputConfiguration']['s3Uri']
job_metadata = get_json_object_from_s3_uri(job_metadata_s3_uri)
#moderated_result = json.loads(object_content)
for segment in job_metadata['output_metadata']:
    job_s3_uri = segment['segment_metadata'][0]['standard_output_path']
    job_output = get_json_object_from_s3_uri(job_s3_uri)
    print(json.dumps(job_output['image'], indent=2))